# Scanpy PBMC 3K profiling (ipynb)

In [1]:
import cProfile
import json
import os
from pathlib import Path
import time

# Equivalent to: python scanpy_pbmc.py --data-set pbmc3k
data_dir = Path("data")
data_set = "pbmc3k"
out_dir = Path("results")
profile_dir = Path("profiles")
num_threads = 1

for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMBA_NUM_THREADS"):
    os.environ[variable] = str(num_threads)

import scanpy as sc

t0 = time.time()

In [2]:
sc.settings.verbosity = 2
sc.settings.n_jobs = num_threads

time_time_timings = {}
t1 = time.time()
time_time_timings["setup"] = t1 - t0
print(f"Section 1 took {t1 - t0:.4f}s")

out_dir.mkdir(parents=True, exist_ok=True)
profile_dir.mkdir(parents=True, exist_ok=True)
timings = {}

def section(name, function):
    start = time.perf_counter()
    result = function()
    elapsed = time.perf_counter() - start
    timings[name] = elapsed
    print(f"SECTION_TIMING dataset={data_set} section={name} seconds={elapsed:.4f}", flush=True)
    return result

print(f"RUN_METADATA dataset={data_set} threads={num_threads}", flush=True)

t2 = time.time()
time_time_timings["configuration"] = t2 - t1
print(f"Section 2 took {t2 - t1:.4f}s")

Section 1 took 0.0096s
RUN_METADATA dataset=pbmc3k threads=1
Section 2 took 0.0014s


## Read the 10x matrix

The on-disk Scanpy cache is disabled so the measurement includes Matrix Market input I/O.

In [3]:
input_dir = data_dir / data_set / "filtered_gene_bc_matrices"
adata = section(
    "read_10x",
    lambda: sc.read_10x_mtx(input_dir, var_names="gene_symbols", cache=False),
)
adata.var_names_make_unique()
print(f"DATASET_SHAPE dataset={data_set} cells={adata.n_obs} genes={adata.n_vars}", flush=True)

t3 = time.time()
time_time_timings["read_10x"] = t3 - t2
print(f"Section 3 took {t3 - t2:.4f}s")
adata

SECTION_TIMING dataset=pbmc3k section=read_10x seconds=0.4422
DATASET_SHAPE dataset=pbmc3k cells=2700 genes=32738
Section 3 took 0.4513s


AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'

## Preprocess

In [4]:
section(
    "filter",
    lambda: (
        sc.pp.filter_cells(adata, min_genes=200),
        sc.pp.filter_genes(adata, min_cells=3),
    ),
)

t4 = time.time()
time_time_timings["filter"] = t4 - t3
print(f"Section 4 took {t4 - t3:.4f}s")

section(
    "normalize_log1p",
    lambda: (
        sc.pp.normalize_total(adata, target_sum=1e4),
        sc.pp.log1p(adata),
    ),
)

t5 = time.time()
time_time_timings["normalize_log1p"] = t5 - t4
print(f"Section 5 took {t5 - t4:.4f}s")

filtered out 19024 genes that are detected in less than 3 cells
SECTION_TIMING dataset=pbmc3k section=filter seconds=0.0404
Section 4 took 0.0512s
normalizing counts per cell
    finished (0:00:02)
SECTION_TIMING dataset=pbmc3k section=normalize_log1p seconds=2.5851
Section 5 took 2.5858s


In [5]:
def select_hvg():
    global adata
    sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)
    adata.raw = adata
    adata = adata[:, adata.var.highly_variable]

section("highly_variable_genes", select_hvg)

t6 = time.time()
time_time_timings["highly_variable_genes"] = t6 - t5
print(f"Section 6 took {t6 - t5:.4f}s")
adata

extracting highly variable genes
    finished (0:00:01)
SECTION_TIMING dataset=pbmc3k section=highly_variable_genes seconds=1.6818
Section 6 took 1.6886s


View of AnnData object with n_obs × n_vars = 2700 × 2000
    obs: 'n_genes'
    var: 'gene_ids', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg'

## Scale, reduce dimensions, construct the graph, and cluster

In [6]:
section("scale", lambda: sc.pp.scale(adata))
t7 = time.time()
time_time_timings["scale"] = t7 - t6
print(f"Section 7 took {t7 - t6:.4f}s")

section("pca", lambda: sc.tl.pca(adata, svd_solver="arpack", n_comps=30))
t8 = time.time()
time_time_timings["pca"] = t8 - t7
print(f"Section 8 took {t8 - t7:.4f}s")

section("neighbors", lambda: sc.pp.neighbors(adata, n_pcs=30))
t9 = time.time()
time_time_timings["neighbors"] = t9 - t8
print(f"Section 9 took {t9 - t8:.4f}s")

section("louvain", lambda: sc.tl.louvain(adata, resolution=0.5))
t10 = time.time()
time_time_timings["louvain"] = t10 - t9
print(f"Section 10 took {t10 - t9:.4f}s")

section("umap", lambda: sc.tl.umap(adata, n_components=30))
t11 = time.time()
time_time_timings["umap"] = t11 - t10
print(f"Section 11 took {t11 - t10:.4f}s")

SECTION_TIMING dataset=pbmc3k section=scale seconds=0.0449
Section 7 took 0.0537s
computing PCA
    with n_comps=30


/opt/scratchspace/shenghan/miniconda3_clean/envs/bmi500/lib/python3.12/functools.py:909: UserWarning: Received a view of an AnnData. Making a copy.
  return dispatch(args[0].__class__)(*args, **kw)
/opt/scratchspace/shenghan/miniconda3_clean/envs/bmi500/lib/python3.12/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


    finished (0:00:00)
SECTION_TIMING dataset=pbmc3k section=pca seconds=0.7249
Section 8 took 0.7254s
computing neighbors
    using 'X_pca' with n_pcs = 30


/opt/scratchspace/shenghan/miniconda3_clean/envs/bmi500/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished (0:00:09)
SECTION_TIMING dataset=pbmc3k section=neighbors seconds=9.1541
Section 9 took 9.1545s
running Louvain clustering
    using the "louvain" package of Traag (2017)


/opt/scratchspace/shenghan/miniconda3_clean/envs/bmi500/lib/python3.12/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: The function louvain is deprecated and will be removed in the future. Use :func:`scanpy.tl.leiden` instead.
  return fn(*args_all, **kw)
/opt/scratchspace/shenghan/miniconda3_clean/envs/bmi500/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


    finished (0:00:00)
SECTION_TIMING dataset=pbmc3k section=louvain seconds=0.2414
Section 10 took 0.2418s
computing UMAP
    finished (0:00:05)
SECTION_TIMING dataset=pbmc3k section=umap seconds=5.9621
Section 11 took 5.9625s


## Save the result and profile marker-gene ranking

In [7]:
output_path = out_dir / f"{data_set}.scanpy.h5ad"
section("write_h5ad", lambda: adata.write(output_path))

t12 = time.time()
time_time_timings["write_h5ad"] = t12 - t11
print(f"Section 12 took {t12 - t11:.4f}s")
output_path

SECTION_TIMING dataset=pbmc3k section=write_h5ad seconds=0.1796
Section 12 took 0.1853s


PosixPath('results/pbmc3k.scanpy.h5ad')

In [8]:
profile_path = profile_dir / f"rank_genes_{data_set}_profile.prof"

def rank_genes():
    profiler = cProfile.Profile()
    profiler.enable()
    sc.tl.rank_genes_groups(adata, "louvain", method="wilcoxon", use_raw=True)
    profiler.disable()
    profiler.dump_stats(profile_path)

section("rank_gene_groups", rank_genes)

t13 = time.time()
time_time_timings["rank_gene_groups"] = t13 - t12
print(f"Section 13 took {t13 - t12:.4f}s")
profile_path

ranking genes
    finished (0:00:11)
SECTION_TIMING dataset=pbmc3k section=rank_gene_groups seconds=11.0666
Section 13 took 11.0840s


PosixPath('profiles/rank_genes_pbmc3k_profile.prof')

In [9]:
print(
    "TIMINGS_JSON "
    + json.dumps({"dataset": data_set, "sections": timings}, sort_keys=True),
    flush=True,
)
print("TIME_TIME_TIMINGS_JSON " + json.dumps({"dataset": data_set, "sections": time_time_timings}, sort_keys=True), flush=True)
print(f"PROFILE_OUTPUT {profile_path}", flush=True)
{"perf_counter": timings, "time_time": time_time_timings}

TIMINGS_JSON {"dataset": "pbmc3k", "sections": {"filter": 0.0403665816411376, "highly_variable_genes": 1.6818107357248664, "louvain": 0.2413518950343132, "neighbors": 9.154052879661322, "normalize_log1p": 2.5851389691233635, "pca": 0.7249070210382342, "rank_gene_groups": 11.066568826325238, "read_10x": 0.44224422704428434, "scale": 0.044937471859157085, "umap": 5.9620911767706275, "write_h5ad": 0.1796481693163514}}
TIME_TIME_TIMINGS_JSON {"dataset": "pbmc3k", "sections": {"configuration": 0.0014324188232421875, "filter": 0.05121660232543945, "highly_variable_genes": 1.688593864440918, "louvain": 0.2417919635772705, "neighbors": 9.154494762420654, "normalize_log1p": 2.5857815742492676, "pca": 0.7253632545471191, "rank_gene_groups": 11.083954811096191, "read_10x": 0.45131802558898926, "scale": 0.053748369216918945, "setup": 0.009555339813232422, "umap": 5.962509870529175, "write_h5ad": 0.18529272079467773}}
PROFILE_OUTPUT profiles/rank_genes_pbmc3k_profile.prof


{'perf_counter': {'read_10x': 0.44224422704428434,
  'filter': 0.0403665816411376,
  'normalize_log1p': 2.5851389691233635,
  'highly_variable_genes': 1.6818107357248664,
  'scale': 0.044937471859157085,
  'pca': 0.7249070210382342,
  'neighbors': 9.154052879661322,
  'louvain': 0.2413518950343132,
  'umap': 5.9620911767706275,
  'write_h5ad': 0.1796481693163514,
  'rank_gene_groups': 11.066568826325238},
 'time_time': {'setup': 0.009555339813232422,
  'configuration': 0.0014324188232421875,
  'read_10x': 0.45131802558898926,
  'filter': 0.05121660232543945,
  'normalize_log1p': 2.5857815742492676,
  'highly_variable_genes': 1.688593864440918,
  'scale': 0.053748369216918945,
  'pca': 0.7253632545471191,
  'neighbors': 9.154494762420654,
  'louvain': 0.2417919635772705,
  'umap': 5.962509870529175,
  'write_h5ad': 0.18529272079467773,
  'rank_gene_groups': 11.083954811096191}}